### Chapter 5.7 - Predicting House Prices on Kaggle

This notebook teaches the Kaggle-style workflow using tiny offline tabular data. It avoids downloads while preserving the real steps: data access, preprocessing, validation, model selection, and submission formatting.


In [ ]:
import math
import random
import numpy as np
import torch


### 5.7.1 Downloading Data

#### 1. Intuition

Kaggle is a platform for data science competitions. Competitions provide training data, test data, and a required submission format.

Downloading data means obtaining those files locally before training.

#### 2. Why this exists

A reproducible project needs to know where data came from and how it was stored. This notebook uses toy offline data instead of network downloads.


#### 3. Examples

Represent downloaded train and test tables as Python rows.


In [ ]:
train_rows = [
    {"id": 1, "rooms": 2.0, "area": 80.0, "price": 200.0},
    {"id": 2, "rooms": 3.0, "area": 120.0, "price": 300.0},
]
test_rows = [{"id": 3, "rooms": 2.0, "area": 100.0}]
len(train_rows), len(test_rows)


#### 4. Step-by-step breakdown

Each dictionary is one row.

Training rows include the label `price`.

Test rows omit `price` because that is what the competition asks us to predict.

The toy data stands in for downloaded CSV files.

#### 5. Connection to ML systems

In a real Kaggle run, this step would read files such as `train.csv` and `test.csv` from disk after downloading them.

#### 6. Common confusion points

- Do not assume network access inside teaching notebooks.
- Training data has labels; competition test data usually does not.
- Keep file paths and data versions documented.
- Toy data is for workflow understanding, not leaderboard performance.


### 5.7.2 Kaggle

#### 1. Intuition

A Kaggle competition evaluates predictions submitted in a required file format.

A leaderboard ranks submissions by a chosen metric. A metric is a rule for scoring predictions.

#### 2. Why this exists

Kaggle adds practical constraints: train locally, predict on hidden-label test data, submit in the expected format, and avoid overfitting the public leaderboard.


#### 3. Examples

A tiny submission row has an ID and prediction.


In [ ]:
submission = [{"Id": 3, "SalePrice": 250.0}]
required_columns = ["Id", "SalePrice"]
list(submission[0].keys()) == required_columns


#### 4. Step-by-step breakdown

The submission dictionary has exactly two keys.

`Id` identifies the test example.

`SalePrice` stores the predicted target value.

The column-order check mirrors the idea of matching a competition format.

#### 5. Connection to ML systems

Real Kaggle submissions are usually CSV files with exactly the required columns and one row per test example.

#### 6. Common confusion points

- The leaderboard score is not the same as local validation performance.
- Public leaderboard feedback can be overused.
- Submission format mistakes can fail even if the model is reasonable.
- Competition test labels are hidden from participants.


### 5.7.3 Accessing and Reading the Dataset

#### 1. Intuition

Accessing data means loading it into memory. Reading a dataset means separating features, labels, and IDs.

For house prices, features might include rooms and area. The label is the sale price.

#### 2. Why this exists

The model should train on features and labels, while IDs are kept for submission but not used as predictive features by default.


#### 3. Examples

Extract feature and label tensors from toy rows.


In [ ]:
X = torch.tensor([[row["rooms"], row["area"]] for row in train_rows])
y = torch.tensor([row["price"] for row in train_rows]).reshape(-1, 1)
test_X = torch.tensor([[row["rooms"], row["area"]] for row in test_rows])
test_ids = [row["id"] for row in test_rows]
X.shape, y.shape, test_X.shape, test_ids


#### 4. Step-by-step breakdown

The list comprehension reads selected feature columns.

`y` stores training prices as a column tensor.

`test_X` stores test features.

`test_ids` stores IDs for later submission.

#### 5. Connection to ML systems

Tabular ML pipelines usually track IDs separately from model inputs.

#### 6. Common confusion points

- IDs identify rows; they are not automatically useful features.
- Feature order must be consistent between train and test data.
- Labels are available for training rows only.
- Shape checks catch many data-loading errors.


### 5.7.4 Data Preprocessing

#### 1. Intuition

Preprocessing prepares raw features for learning.

Common tabular preprocessing includes filling missing values, standardizing numerical features, and encoding categorical features.

#### 2. Why this exists

Features with very different scales can make optimization harder. Missing or categorical values must be handled before tensor training.


#### 3. Examples

Standardize numerical features.


In [ ]:
mean = X.mean(dim=0, keepdim=True)
std = X.std(dim=0, keepdim=True)
X_scaled = (X - mean) / std
test_scaled = (test_X - mean) / std
X_scaled, test_scaled


#### 4. Step-by-step breakdown

`mean` stores the training feature averages.

`std` stores the training feature standard deviations.

Training and test features use the same training mean and standard deviation.

Using test statistics would leak information from the test set.

#### 5. Connection to ML systems

Real Kaggle preprocessing often combines numerical standardization and one-hot encoding for categorical columns.

#### 6. Common confusion points

- Fit preprocessing values on training data only.
- Apply the same transformation to validation and test data.
- Categorical features need encoding.
- Preprocessing choices should be reproducible.


### 5.7.5 Error Measure

#### 1. Intuition

House-price competitions often use root mean squared log error, or RMSLE.

A logarithm reduces the dominance of large prices and measures relative error more than absolute error.

#### 2. Why this exists

Predicting 10 percent too high on a small house and 10 percent too high on a large house should be treated more similarly than raw dollar errors would.


#### 3. Examples

Compute log-RMSE for positive predictions and labels.


In [ ]:
pred = torch.tensor([[210.0], [280.0]])
label = torch.tensor([[200.0], [300.0]])
log_error = torch.log(pred) - torch.log(label)
rmse_log = torch.sqrt((log_error ** 2).mean())
rmse_log


#### 4. Step-by-step breakdown

`torch.log` takes the natural logarithm.

The difference compares log predictions with log labels.

Squaring makes errors positive.

The mean and square root produce root mean squared log error.

#### 5. Connection to ML systems

The evaluation metric should guide validation and model selection.

#### 6. Common confusion points

- Log metrics require positive values.
- The competition metric may differ from the training loss.
- Relative error matters more under log metrics.
- Clamp or constrain predictions if they might become nonpositive.


### 5.7.6 k-Fold Cross-Validation

#### 1. Intuition

k-fold cross-validation splits training data into `k` parts. Each part becomes validation data once while the others are used for training.

The final validation estimate averages across folds.

#### 2. Why this exists

Cross-validation uses limited data more efficiently than one fixed validation split.


#### 3. Examples

Create fold indices for a tiny dataset.


In [ ]:
n = 6
k = 3
indices = torch.arange(n)
fold_size = n // k
folds = [indices[i * fold_size:(i + 1) * fold_size] for i in range(k)]
folds


#### 4. Step-by-step breakdown

`n` is the number of examples.

`k` is the number of folds.

`fold_size` is the number of examples per fold in this simple case.

Each slice becomes one validation fold.

#### 5. Connection to ML systems

Kaggle solutions often use cross-validation to compare model settings before producing test predictions.

#### 6. Common confusion points

- Folds should preserve row-label alignment.
- Real datasets may not divide evenly by `k`.
- Cross-validation is more expensive than one split.
- Test data should not be part of cross-validation.


### 5.7.7 Model Selection

#### 1. Intuition

Model selection means choosing hyperparameters or model designs using validation performance.

A hyperparameter is a setting chosen outside gradient training, such as hidden size, learning rate, or weight decay.

#### 2. Why this exists

Different settings can produce different validation scores, so selection should be systematic.


#### 3. Examples

Choose the model setting with the lowest validation score.


In [ ]:
runs = [
    {"hidden": 8, "score": 0.18},
    {"hidden": 16, "score": 0.15},
    {"hidden": 32, "score": 0.17},
]
best = min(runs, key=lambda run: run["score"])
best


#### 4. Step-by-step breakdown

Each dictionary stores one candidate setting and validation score.

`min` chooses the smallest score because lower error is better.

The selected hidden size is chosen by validation performance.

#### 5. Connection to ML systems

Good Kaggle workflows record tried settings and avoid repeatedly tuning on public leaderboard feedback.

#### 6. Common confusion points

- Use validation or cross-validation for selection.
- Keep the test submission process separate.
- Hyperparameters are not ordinary learned weights.
- Record experiments so results are traceable.


### 5.7.8 Submitting Predictions on Kaggle

#### 1. Intuition

Submitting predictions means creating a file with test IDs and predicted target values in the required format.

The model predicts labels for test rows whose true labels are hidden.

#### 2. Why this exists

A correct submission format is required before scoring can happen.


#### 3. Examples

Create toy submission rows from IDs and predictions.


In [ ]:
test_pred = torch.tensor([[260.0]])
submission = []
for row_id, pred_value in zip(test_ids, test_pred.reshape(-1)):
    submission.append({"Id": row_id, "SalePrice": float(pred_value)})
submission


#### 4. Step-by-step breakdown

`test_pred` stores predicted prices for test examples.

`zip(test_ids, test_pred.reshape(-1))` pairs each ID with one prediction.

Each output dictionary has the required submission fields.

Real code would write these rows to a CSV file.

#### 5. Connection to ML systems

The submission step is deployment-like: the model produces outputs for examples without known labels.

#### 6. Common confusion points

- The number of predictions must match the number of test IDs.
- Submission column names must match the competition requirement.
- Predictions should be in the metric's valid range.
- Do not include training labels in the submission.


### 5.7.9 Summary and Discussion

#### 1. Intuition

A Kaggle workflow includes data access, preprocessing, local validation, model selection, test prediction, and submission formatting.

#### 2. Why this exists

The competition setting rewards both modeling and careful engineering discipline.


#### 3. Examples

A compact Kaggle workflow checklist.


In [ ]:
workflow = [
    "read train/test data",
    "preprocess using train statistics",
    "validate models locally",
    "predict test labels",
    "write submission",
]
workflow


#### 4. Step-by-step breakdown

The checklist names the required workflow order.

Preprocessing must be fit on training data.

Validation should guide model choices.

Submission happens after choosing the model.

#### 5. Connection to ML systems

This workflow is also useful outside Kaggle whenever test labels are unavailable at prediction time.

#### 6. Common confusion points

- Kaggle score is only as meaningful as the competition setup.
- Public leaderboard tuning can overfit.
- Reproducibility matters.
- Offline toy examples teach workflow, not leaderboard strategy.


### 5.7.10 Exercises

#### 1. Intuition

These exercises practice the mechanics of a Kaggle-style tabular workflow.

#### 2. Why this exists

Small offline examples help separate workflow understanding from competition logistics.


#### 3. Examples

Exercise 1: standardize a new feature matrix.


In [ ]:
X2 = torch.tensor([[1.0, 10.0], [3.0, 30.0]])
mean2 = X2.mean(dim=0, keepdim=True)
std2 = X2.std(dim=0, keepdim=True)
(X2 - mean2) / std2


Exercise 2: build a two-row submission list.


In [ ]:
ids = [10, 11]
preds = torch.tensor([100.0, 120.0])
[{"Id": i, "SalePrice": float(p)} for i, p in zip(ids, preds)]


#### 4. Step-by-step breakdown

Exercise 1 checks numerical preprocessing.

Exercise 2 checks submission formatting.

Both operations are common in tabular competitions.

#### 5. Connection to ML systems

These mechanics support the larger model-training workflow.

#### 6. Common confusion points

- Use training statistics for scaling.
- Preserve row ID order for submissions.
- Predictions should be numeric.
- Keep competition logistics separate from model concepts.
